# Etapa B - Python / Pandas

Dataset 5 - Wind & Solar Energy Production (Kaggle)

A filtragem e a amostragem de 20% foram feitas no Orange na Etapa A. Aqui uso o arquivo exportado
de lá, `energy_production_filter.csv`.

O objetivo é comparar com que frequência cada fonte trabalha acima de 70% do seu próprio máximo.
Como solar e eólica têm escalas diferentes, cada uma é comparada com o máximo dela mesma.

No arquivo a produção das duas fontes está na mesma coluna `Production`, separada pela coluna
`Source`.

In [2]:
import pandas as pd

df = pd.read_csv("/content/energy_production_filter.csv")

print(df.shape)
df.head()

(10373, 4)


,Date,Source,Season,Production
0,2022-12-18 00:00:00,Wind,Winter,11400
1,2024-01-29 00:00:00,Wind,Winter,7917
2,2024-08-01 00:00:00,Solar,Summer,8835
3,2020-11-10 00:00:00,Wind,Fall,989
4,2022-12-19 00:00:00,Wind,Winter,12526


In [3]:
# B.1 - separando as fontes e renomeando as variáveis
solar = df[df["Source"] == "Solar"].rename(columns={"Production": "Geracao_Solar"})
eolica = df[df["Source"] == "Wind"].rename(columns={"Production": "Geracao_Eolica"})

print("Solar:", len(solar), "registros")
print("Eólica:", len(eolica), "registros")
solar.head()

Solar: 1846 registros
Eólica: 8527 registros


,Date,Source,Season,Geracao_Solar
2,2024-08-01 00:00:00,Solar,Summer,8835
5,2024-10-18 00:00:00,Solar,Fall,4855
6,2020-04-10 00:00:00,Solar,Spring,3692
13,2021-06-02 00:00:00,Solar,Summer,5561
16,2021-05-02 00:00:00,Solar,Spring,4993


In [4]:
# B.2 - máximo de cada fonte
max_solar = solar["Geracao_Solar"].max()
max_eolica = eolica["Geracao_Eolica"].max()

print("Máximo solar:", max_solar)
print("Máximo eólico:", max_eolica)

Máximo solar: 16316
Máximo eólico: 22929


In [5]:
# B.3 - 70% do máximo de cada fonte
limiar_solar = 0.7 * max_solar
limiar_eolica = 0.7 * max_eolica

print("Limiar solar:", limiar_solar)
print("Limiar eólico:", limiar_eolica)

Limiar solar: 11421.199999999999
Limiar eólico: 16050.3


In [6]:
# B.4 - DataFrames de alta geração
alta_solar = solar[solar["Geracao_Solar"] >= limiar_solar]
alta_eolica = eolica[eolica["Geracao_Eolica"] >= limiar_eolica]

print("Alta geração solar:", len(alta_solar))
print("Alta geração eólica:", len(alta_eolica))
alta_eolica.head()

Alta geração solar: 45
Alta geração eólica: 245


,Date,Source,Season,Geracao_Eolica
20,2024-01-03 00:00:00,Wind,Winter,17272
32,2021-03-11 00:00:00,Wind,Spring,16144
103,2025-01-29 00:00:00,Wind,Winter,18199
151,2024-12-08 00:00:00,Wind,Winter,16946
192,2024-11-19 00:00:00,Wind,Fall,16568


In [7]:
# B.5 - contagem e percentuais
n_solar = len(alta_solar)
n_eolica = len(alta_eolica)
total = len(df)

# percentual sobre o total de registros
pct_total_solar = n_solar / total * 100
pct_total_eolica = n_eolica / total * 100

# percentual sobre os registros de cada fonte
pct_solar = n_solar / len(solar) * 100
pct_eolica = n_eolica / len(eolica) * 100

resultado = pd.DataFrame({
    "Fonte": ["Solar", "Eólica"],
    "Maximo": [max_solar, max_eolica],
    "Limiar_70": [limiar_solar, limiar_eolica],
    "Acima_do_limiar": [n_solar, n_eolica],
    "Total_da_fonte": [len(solar), len(eolica)],
    "Pct_sobre_total": [round(pct_total_solar, 2), round(pct_total_eolica, 2)],
    "Pct_sobre_a_fonte": [round(pct_solar, 2), round(pct_eolica, 2)]
})
print("Total de registros:", total)
resultado

Total de registros: 10373


,Fonte,Maximo,Limiar_70,Acima_do_limiar,Total_da_fonte,Pct_sobre_total,Pct_sobre_a_fonte
0,Solar,16316,11421.2,45,1846,0.43,2.44
1,Eólica,22929,16050.3,245,8527,2.36,2.87


In [8]:
# B.6 - comparação
print("Percentual sobre o total de registros:")
print("  Solar: {:.2f}%  Eólica: {:.2f}%".format(pct_total_solar, pct_total_eolica))
print()
print("Percentual sobre os registros de cada fonte:")
print("  Solar: {:.2f}%  Eólica: {:.2f}%".format(pct_solar, pct_eolica))
print()

if pct_eolica > pct_solar:
    print("A eólica fica acima de 70% do próprio máximo com mais frequência.")
else:
    print("A solar fica acima de 70% do próprio máximo com mais frequência.")

Percentual sobre o total de registros:
  Solar: 0.43%  Eólica: 2.36%

Percentual sobre os registros de cada fonte:
  Solar: 2.44%  Eólica: 2.87%

A eólica fica acima de 70% do próprio máximo com mais frequência.


## Respostas

### B.6 - Qual fonte aparece com maior frequência acima de 70% do seu próprio máximo?

A eólica. Foram 245 registros de alta geração eólica contra 45 de alta geração solar.

Calculei o percentual de duas formas porque o resultado muda bastante:

- Sobre o total de registros: 2,36% para a eólica e 0,43% para a solar.
- Sobre os registros de cada fonte: 2,87% para a eólica e 2,44% para a solar.

A primeira forma exagera a diferença. A solar representa só 17,8% das linhas do dataset, porque no
arquivo original ela só é medida durante o dia, enquanto a eólica é medida nas 24 horas. Então
dividir pelo total de registros acaba penalizando a solar por uma característica do dataset, e não
pelo desempenho dela. A segunda forma é a mais justa, já que o enunciado pede para comparar cada
fonte com ela mesma.

De qualquer maneira a eólica ganha nas duas contas, então a resposta não depende dessa escolha.

A explicação para isso é o comportamento das duas fontes. O vento é irregular, mas quando venta
forte ele venta por várias horas seguidas, e a produção fica perto do máximo durante todo esse
período. Já a solar segue sempre o mesmo ciclo diário, e o máximo dela (16.316) só acontece em
condições raras, tipo meio-dia de verão com céu limpo. Ou seja, a solar é mais previsível, mas
justamente por isso passa pouco tempo perto do próprio pico.

Vale observar que este arquivo tem só uma fonte por linha, então não dá para verificar se as duas
fontes atingem o pico ao mesmo tempo.

### B.7 - Por que não usar o mesmo valor numérico como limite para as duas fontes?

Porque as duas fontes têm escalas diferentes, e um limite fixo compararia o tamanho dos parques em
vez do desempenho de cada um.

Neste dataset o erro é fácil de cometer, já que as duas produções estão na mesma coluna. Se eu
tivesse feito `df["Production"].max()` sem separar por fonte, o resultado seria 22.929, que é o
máximo da eólica. Usando 70% disso (16.050) para as duas, a solar quase nunca apareceria como alta
geração, porque o máximo dela é 16.316. O número mostraria só que o parque eólico é maior, e não
com que frequência cada fonte trabalha perto do limite dela.

Dividindo pelo máximo de cada uma, o valor deixa de ser em unidade de potência e vira uma fração
da capacidade observada. Aí sim as duas ficam comparáveis. É a mesma ideia do fator de capacidade,
que serve para comparar usinas de portes diferentes.

Outro motivo é que as distribuições têm formatos diferentes. A solar é limitada pelo ciclo do dia e
a eólica é mais espalhada, com valores altos ocasionais. Um mesmo corte cairia em pontos bem
diferentes de cada distribuição.

Por último, uma observação sobre o método: usar o máximo observado como referência deixa o limiar
sensível a valores extremos, e também ao fato de eu estar trabalhando com 20% dos dados. Uma opção
mais estável seria usar a capacidade instalada de cada parque ou um percentil alto, como o P99.